# Home Credit Analysis Runner

Thin notebook wrapper for the analysis stage in `src/hc_risk/`.

Start with the sampled cached run first. A full `FORCE=True` rebuild with no sampling can take a while.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "hc_risk").exists() and (ROOT.parent / "src" / "hc_risk").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "hc_risk").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
import json
from pprint import pprint

from hc_risk.analysis import run_analysis
from hc_risk.config import PipelineConfig

In [ ]:
FORCE = False
ANALYSIS_BASELINE_SAMPLE_ROWS = 10000

config = PipelineConfig(
    analysis_baseline_sample_rows=ANALYSIS_BASELINE_SAMPLE_ROWS,
)
config

In [ ]:
print(f"Starting analysis run with force={FORCE}, baseline_sample_rows={ANALYSIS_BASELINE_SAMPLE_ROWS}")
artifacts = run_analysis(config, force=FORCE)

In [ ]:
with (config.reports_dir / "analysis_baseline_metrics.json").open() as handle:
    baseline_metrics = json.load(handle)
with (config.reports_dir / "target_analysis.json").open() as handle:
    target_analysis = json.load(handle)

summary = {
    "master_rows": len(artifacts.master_bundle.master),
    "train_rows": int((artifacts.master_bundle.master["dataset_split"] == "train").sum()),
    "test_rows": int((artifacts.master_bundle.master["dataset_split"] == "test").sum()),
    "leaderboard_features": len(artifacts.master_bundle.leaderboard_features),
    "categorical_features": len(artifacts.master_bundle.categorical_features),
    "analysis_baseline_roc_auc": baseline_metrics["roc_auc"],
    "analysis_baseline_pr_auc": baseline_metrics["pr_auc"],
    "top_numeric_predictors": target_analysis["top_numeric_predictors"][:5],
}
pprint(summary)

In [ ]:
print("Reports written under:", config.reports_dir)
print("Artifacts written under:", config.outputs_dir)